In [ ]:
import random
from time import sleep, time

from distributed_processing.client import Client
from distributed_processing.redis_connector import RedisConnector

In [ ]:
REDIS_HOST = "localhost"
REDIS_PORT = 6379
REDIS_DB = 0
NAMESPACE = "tasks"

In [ ]:
redis_connector = RedisConnector(
    redis_host=REDIS_HOST, redis_port=REDIS_PORT, redis_db=REDIS_DB, namespace=NAMESPACE
)

client = Client(redis_connector, check_registry="cache")

In [ ]:
y = client.rpc_async("add", [1, 0])

In [ ]:
y.get()

In [ ]:
client.check_registry = "always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

In [ ]:
client.check_registry = "never"

client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


In [ ]:
client.check_registry = "never"

client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

In [ ]:
client.check_registry = "cache"

In [ ]:
x = client.rpc_async("div", [1, 0])

In [ ]:
try:
    x.get()
except Exception as e:
    print(e)

In [ ]:
client.rpc_sync("add", [1, 1])

In [ ]:
def f(x, y):
    return x + y


y = client.rpc_async_fn(f, [1, 2.0])

In [ ]:
y.get()

In [ ]:
client.rpc_sync_fn(f, [3.0, 2.0])

In [ ]:
# Fan out N requests to random methods; each one returns an AsyncResult.
# lista/tupla/dic show how containers travel: with the default JsonSerializer
# a tuple reaches the client as a list.
fs = []
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(), random.random()], {})
    print(t)
    tp.append(t)
    fs.append(client.rpc_async(t[0], t[1]))

In [ ]:
[f.get() for f in fs]

In [ ]:
# rpc_multi_async sends the same list of (method, args, kwargs) as independent
# requests, spread across the workers. Compare with rpc_batch_async below:
# one message, one worker.
fs = client.rpc_multi_async(tp)

In [ ]:
# AsyncResult.status checks the responses queue every time it is read.
# 'PENDING' is transitory: if the response is already in the queue or in the
# client cache, reading status updates the object.
[f.status for f in fs]

In [ ]:
# Wait at most 2 s for every pending response. Returns the ids still pending
# ([] when all responses have arrived).
client.wait_responses(timeout=2)

In [ ]:
# Without timeout waits (up to client.timeout) for every pending response.
client.wait_responses()

In [ ]:
[f.status for f in fs]

In [ ]:
# Unknown ids (neither pending nor cached) raise ValueError.
try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

In [ ]:
# Pending ids: {id: time the request was sent}. Empty once everything arrived.
client.pending

In [ ]:
[f.get() for f in fs]

In [ ]:
tp

In [ ]:
fs = client.rpc_batch_async(tp)

In [ ]:
[f.get() for f in fs]

In [ ]:
client.rpc_batch_sync(tp)

In [ ]:
# A batch that includes "fake", a method no worker offers, on purpose: the
# next cells show how each check_registry mode handles it.
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

In [ ]:
tp

In [ ]:
# A batch is one message run by one worker, so the client needs a queue that
# serves every method in it. With check_registry="never" every method maps
# to the default queue without consulting the registry: the batch is sent
# and the worker answers request by request.
client.check_registry = "never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [ ]:
# Results for add/mul/div; None for "fake" (the worker answered -32601,
# method not found, and safe_get hides the error).
[f.safe_get() for f in fs]

In [ ]:
# Same as above, but the queue is passed explicitly. An explicit queue is used
# as is in any mode (as with rpc_async): the registry is not consulted, the
# default queue does not matter, and the worker answers -32601 for "fake".
client.check_registry = "never"
client.set_default_queue("")

fs = client.rpc_batch_async(tp, queue="cola_1")
[f.safe_get() for f in fs]

In [ ]:
# With check_registry="cache" (or "always") the queues come from the registry.
# Refreshing the cache changes nothing: the registry is right, "fake" is not
# in it.
client.check_registry = "cache"
client.update_registry_cache()

In [ ]:
# "fake" has no queue, the intersection of queues is empty and the client
# raises ValueError before sending anything: the whole batch is refused, since
# no single worker could run all of it. rpc_multi_async would instead send the
# requests one by one and fail only for the unknown method.
try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

In [ ]:
# The rule holds in "cache"/"always" too: an explicit queue is used as is, the
# registry only decides when no queue is given. The batch goes to cola_1 and
# the worker answers -32601 for "fake" (None with safe_get).
client.rpc_batch_sync(tp, timeout=5, queue="cola_1")

In [ ]:
client.check_registry = "never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [ ]:
try:
    x.get()
except Exception as e:
    print(e)

In [ ]:
y = client.rpc_async("add", [1, 0])

In [ ]:
y.get(5)

In [ ]:
# Sending python function to be evaluated to the default queue.

def f(x, y):
    return x + y


client.check_registry = "never"
y = client.rpc_async_fn(f, [1, 2.0])
try:
    y.get()
except Exception as e:
    print(e)

In [ ]:
# Back to "cache": rpc_async_fn needs the registry to find the py_eval queue.
client.check_registry = "cache"
client.rpc_sync_fn(f, [1, 2.0])

In [ ]:
# Responses cache: get(clean=True) (the default) removes the response once
# read; clean_used() removes every response that has been read at least once.
y = client.rpc_async("add", [1, 1])
y.get(clean=False)
list(client.responses.keys())

In [ ]:
client.clean_used()
list(client.responses.keys())

In [ ]:
# The function runs on the worker: "hola" is printed there, the result is None.
client.rpc_sync_fn(print, ["hola"])

In [ ]:
# Methods on a queue added with register=False are not in the registry, but a
# client that knows the queue name can call them.
# Positional and named parameters can be combined, as in Python. This is a
# deliberate departure from JSON-RPC 2.0, whose params is one array OR one
# object; the worker runs fn(*args, **kwargs).
client.rpc_sync("hola", ["Ana"], {"calificativo": "rápida"}, queue="cola_2")

In [ ]:
# The same parameter given positionally and by name is a clash: the worker
# answers with -32602 (invalid params) instead of running the function.
try:
    client.rpc_sync("hola", ["Ana"], {"nombre": "Eva"}, queue="cola_2")
except Exception as e:
    print(e)

In [ ]:
# Registry seen by the client, with simple queue names. update=True refreshes
# the cache first.
client.registry(update=True)

In [ ]:
client.all_queues_for_method("add"), client.all_workers_for_method("add")

In [ ]:
# retry() only resends a request that is still pending. 
# The request must have been created with retry=True.
fs[0].retry()

In [ ]:
# We can also use the Client method wait_responses to gather the results.
# wait_responses takes the ids of the requests as an optional argument.
# The id of an AsyncResult instance can be accessed with AsyncResult.id.
# If no ids are passed, we wait for all the responses pending on this Client
# instance. With timeout=0 the queue is checked only once, so we only get the
# responses that are already available. 


fs = client.rpc_multi_async([("sleep", [3], {})] * 4, retry=False)
client.wait_responses([f.id for f in fs], timeout=60)

In [ ]:
# ids of the AsyncResult instances returned by rpc_multi_async

[f.id for f in fs]

In [ ]:
[f.status for f in fs], [f.retries for f in fs]

In [ ]:
# If no ids are passed, we wait for all the responses pending on this Client
# instance. With timeout=0 the queue is checked only once, so we only get the
# responses that are already available.
# wait_responses lets us refresh the state of the client's pending responses
# and gives us fine-grained control over which responses we wait for and when.

fs = client.rpc_multi_async([("sleep", [3], {})] * 4, retry=False)

sleep(15)

client.wait_responses(timeout=0)
[f.status for f in fs], [f.retries for f in fs]

In [ ]:
# Execution time on the worker, from the response metadata.
# metadata is always available, it is generated by the worker.
[
    f.metadata["timing"]["execution_finish"] - f.metadata["timing"]["execution_start"]
    for f in fs
]

In [ ]:
fs[0].metadata

In [ ]:
fs[0].creation_time

In [ ]:
# gather is similar to the wait_responses method but works for different clients.
# It takes a list of AsyncResults as argument.
# ones that look lost (older than a completed one in the same queue) when
# there is no progress for a while. Returns True when every response arrived.
from distributed_processing import gather

fs = client.rpc_multi_async([("sleep", [3], {})] * 4, retry=True)
gather(fs, timeout=6)

In [ ]:
[f.status for f in fs], [f.retries for f in fs]

In [ ]:
max([c.timeout for c in set([f._client for f in fs])])

In [ ]:
# Test shared variables
client.variables()

In [ ]:
# Test shared variables
client.get_variable("shared_variable")

In [ ]:
# Test updating shared variables
client.update_variable("shared_variable", lambda x: x + 111)

In [ ]:
# Test updating shared variables
client.get_variable("shared_variable")

In [ ]:
# Test updating shared variables
try:
    client.update_variable("missing_variable", lambda x: x + 111)
except Exception as e:
    print(e)

In [ ]:
# Test updating shared variables
client.update_variable("missing_variable", lambda x: x + 111, 0)

In [ ]:
client.variables()

In [ ]:
client.delete_variable("missing_variable")
client.variables()